# Upgrade a Lista de Entregas

In [1]:
from pathlib import Path
import pandas as pd
import json
import datetime as dt
import re
from difflib import get_close_matches

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
print('Importadas ok')

Importadas ok


# Input Collect Datos

## 1) Lista de Entregas
Fenix

In [2]:
carpeta = "../data/lista_entregas/"
file = carpeta + "CRONOGRAMA_ENTREGAS_FENIX_HARUMI.xlsx"

In [3]:
df_entregas_raw = pd.ExcelFile(file)
print(f'Pestañas: {df_entregas_raw.sheet_names}')


df_entregas = pd.read_excel(file, sheet_name="CRONO-ORD-FECHA", skiprows=2, dtype={'DNI': str})

print(f'Total Entregas Pestaña 1: {df_entregas.shape[0]} clientes')
df_entregas.head()

Pestañas: ['CRONO-ORD-FECHA', 'CRONO-ORD-PISO', 'EN PROCESO', 'LIBRES']
Total Entregas Pestaña 1: 39 clientes


,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN


In [4]:
df_entregas

,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN
5,5.0,DEPARTAMENTO,804,ROMO AGUIRRE LUIGUI ROLANDO,74827647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20 Dic 2024,02 Oct 2025,22 Jul 2026,9:00 am,NaN,NaN
6,6.0,DEPARTAMENTO,704,LLANQUE FRAQUITA MANINO HELAMAN,40654929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Jul 2025,14 Oct 2025,22 Jul 2026,11:00 am,NaN,NaN
7,7.0,DEPARTAMENTO,1705,BULEJE CALDERON WILDER WILFREDO,71477258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,30 Oct 2025,22 Jul 2026,2:00 pm,NaN,NaN
8,8.0,DEPARTAMENTO,303,ALDERETE SOLIS SADITH PATRICIA,46137521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,31 Oct 2025,22 Jul 2026,4:00 pm,NaN,NaN
9,9.0,DEPARTAMENTO,802,BELLIDO PANTOJA ROSA MARIA,73097248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21 Ago 2025,24 Dic 2025,24 Jul 2026,9:00 am,NaN,NaN


In [5]:
df_procesos = pd.read_excel(file, sheet_name="EN PROCESO", skiprows=2)

print(f'Total Procesos Pestaña 2: {df_procesos.shape[0]} clientes')
df_procesos.head()

Total Procesos Pestaña 2: 66 clientes


,#,Tipo,N° Unidad,Comprador,% Avance,F. Separación
0,1,ESTACIONAMIENTO,7,BELLIDO PANTOJA ROSA MARIA,0.770,24 Jun 2025
1,2,DEPARTAMENTO,402,BLAZ MARIÑO FRANK,0.100,06 Jun 2026
2,3,DEPARTAMENTO,1201,CACERES VALVERDE HENRY ENRIQUE,0.003,16 Jun 2026
3,4,DEPARTAMENTO,903,CAMAHUALI HUAMAN NANCY LIZ,0.249,06 Abr 2025
4,5,ESTACIONAMIENTO,32,CAMAHUALI HUAMAN NANCY LIZ,0.000,NaN


In [6]:
import numpy as np

# 1) Asegurar nombres limpios
df_procesos.columns = df_procesos.columns.astype(str).str.strip()

# 2) Limpiar N° Unidad para evitar valores tipo 1201.0
df_procesos["unidad_key"] = (
    df_procesos["N° Unidad"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# 3) Normalizar Tipo
tipo_key = (
    df_procesos["Tipo"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 4) Crear codigo_unidad
df_procesos["codigo_unidad"] = np.where(
    tipo_key.eq("ESTACIONAMIENTO"),
    "FX-E" + df_procesos["unidad_key"],
    "FX-" + df_procesos["unidad_key"]
)

df_procesos[["Tipo", "N° Unidad", "codigo_unidad"]].head()

,Tipo,N° Unidad,codigo_unidad
0,ESTACIONAMIENTO,7,FX-E7
1,DEPARTAMENTO,402,FX-402
2,DEPARTAMENTO,1201,FX-1201
3,DEPARTAMENTO,903,FX-903
4,ESTACIONAMIENTO,32,FX-E32


## 2) Clientes Sperant

In [7]:
carpeta = "../data/raw/"
file = carpeta + "clientes.parquet"

In [8]:
df_raw = pd.read_parquet(file)
df_raw.head()

,fecha_creacion,nombres,apellidos,tipo_documento,documento,genero,estado_civil,email,telefono,celulares,agrupacion_medio_captacion,medio_captacion,canal_entrada,nivel_interes,fecha_nacimiento,nacionalidad,pais,departamento,provincia,distrito,direccion,apto,observacion,ocupacion,documento_conyuge,usuario_creador,username,estado,ultimo_proyecto,total_unidades_asignadas,ultimo_vendedor,total_interacciones,fecha_ultima_interaccion,proyectos_relacionados,id,codigo_externo_cliente,rango_edad,origen,ultimo_tipo_interaccion,fecha_actualizacion,autorizacion_uso_datos,autorizacion_publicidad,geo_latitud,geo_longitud,geolocalizacion,cliente_riesgo,agrupacion_canal_entrada,tipo_persona,denominacion,tipo_financiamiento,desistido,razon_desistimiento,hora_creacion,fecha_primera_interaccion_manual,prioridad,uuid
0,2020-08-18,ANA DELIA,ESPINO PILLPE,DNI,auto-964705,Masculino,NaN,deliaespino82@hotmail.com,975170003,+51975170003,NaN,contacto web,llamada telefónica,por contactar,None,NaN,PE,LIM,NaN,NaN,NaN,Apto,NaN,NaN,NaN,NaN,NaN,Interesado,Sialia,0,Victor Salazar,8,2026-06-30,"NP,SL,TZ,CRUZ",30760,None,None,api,facebook,2026-06-30,True,False,None,None,None,no,None,Persona Natural,NaN,NaN,no,NaN,01:57:05,2023-09-11 20:18:12.103694,None,9b5af177-f48c4c-ed73
1,2026-06-18,Jorge,Arrelucea,DNI,auto-11818158104,Femenino,NaN,jarrelucefernandez@gmail.com,NaN,+51921201738,Digital,facebook,contacto web,por contactar,None,NaN,NaN,NaN,NaN,NaN,NaN,Apto,"s/70,000_a_más",NaN,NaN,NaN,NaN,Interesado,Campañas,0,NaN,2,2026-06-18,CAM,151543,None,None,fblead_ads,facebook,2026-06-18,True,False,None,None,None,no,None,Persona Natural,NaN,NaN,no,NaN,11:16:10,NaT,None,b9593f8c-4c170c-1177
2,2026-05-31,Luis,Lezameta,DNI,auto-2063323679,Femenino,NaN,lezametaroca79@gmail.com,NaN,+51947497507,Digital,facebook,contacto web,por contactar,None,NaN,NaN,NaN,NaN,NaN,NaN,Apto,2_dorm_desde_55.48_m2,NaN,NaN,NaN,NaN,Interesado,Torre Nápoles,0,NaN,3,2026-06-03,NP,149925,None,None,fblead_ads,llamada por teléfono,2026-06-03,True,False,None,None,None,no,None,Persona Natural,NaN,NaN,no,NaN,10:32:16,NaT,None,921fec00-e12716-976b
3,2026-06-18,sofia,uriarte,DNI,auto-128982,Femenino,soltero,suriarteva1@gmail.com,,+51965766611,NaN,paso por la zona,whatsapp,por contactar,None,peruvian,PE,LIM,Lima,NaN,,Apto,,,NaN,Mayra Nahui,mnahui,Interesado,Capadocia,0,NaN,2,2026-06-20,CP,151545,None,None,manual,whatsapp,2026-06-20,True,True,None,None,None,no,None,Persona Natural,,NaN,no,NaN,11:42:39,NaT,None,ddf9717c-10dbb5-16be
4,2026-06-18,Karol Milena,Añez Mondaca,DNI,auto-1542075611,Femenino,NaN,kami_am29@hotmail.com,NaN,+51937395480,Digital,facebook,contacto web,por contactar,None,NaN,NaN,NaN,NaN,NaN,NaN,Apto,3_dorm_desde_65.28_m2,NaN,NaN,NaN,NaN,Interesado,Capadocia,0,NaN,3,2026-06-22,CP,151537,None,None,fblead_ads,whatsapp,2026-06-22,True,False,None,None,None,no,None,Persona Natural,NaN,NaN,no,NaN,09:51:17,NaT,None,0de8e4b9-7704f2-d4cf


### Propietarios

In [9]:
df_propietarios = df_raw[df_raw['estado']=="Propietario"]
print(f'Total Propietarios: {df_propietarios.shape[0]} clientes')
col_contacto = ['documento', 'celulares', 'email', 'telefono', 'nombres', 'apellidos']
df_propietarios = df_propietarios[col_contacto]
df_propietarios
df_propietarios.sample(2)


Total Propietarios: 1301 clientes


,documento,celulares,email,telefono,nombres,apellidos
130860,70557893,+51973244236,fiorella.alvaradof@gmail.com,,FIORELLA DEL CARMEN,ALVARADO FALCON
15763,41950248,+51949915619,jahsgs@hotmail.com,,Jessica María,Alvan Ramirez


### Calidad de celular

In [10]:
def limpiar_celular(valor):
    if pd.isna(valor):
        return pd.NA
    
    s = str(valor).strip()
    
    # Dejar solo números
    s = re.sub(r"\D", "", s)
    
    # Quitar prefijo Perú si viene como 51XXXXXXXXX
    if s.startswith("51") and len(s) == 11:
        s = s[2:]
    
    # Quitar prefijo Perú si viene como 0051XXXXXXXXX
    if s.startswith("0051") and len(s) == 13:
        s = s[4:]
    
    # Si quedó vacío
    if s == "":
        return pd.NA
    
    return s

In [11]:
df_propietarios["celular_clean"] = df_propietarios["celulares"].apply(limpiar_celular)

df_propietarios["celular_len"] = df_propietarios["celular_clean"].astype("string").str.len()

df_propietarios["celular_ok"] = (
    (df_propietarios["celular_len"] == 9) &
    (df_propietarios["celular_clean"].astype("string").str.startswith("9"))
)

In [12]:
df_propietarios["celular_ok"].value_counts(dropna=False)

celular_ok
True     1031
<NA>      145
False     125
Name: count, dtype: Int64

In [13]:
df_propietarios.loc[
    ~df_propietarios["celular_ok"].fillna(False),
    ["celulares", "celular_clean", "celular_len"]
].head(30)

,celulares,celular_clean,celular_len
569,+33610760107,33610760107,11
710,NaN,NaN,<NA>
6206,NaN,NaN,<NA>
8265,+11997888605,11997888605,11
8668,+5199156508,5199156508,10
10607,NaN,NaN,<NA>
10669,+511234567,511234567,9
11069,NaN,NaN,<NA>
13272,+5165168168168161681,5165168168168161681,19
13311,+516830617681,516830617681,12


### Calidad de Email

In [14]:
DOMINIOS_CORRECTOS = [
    "gmail.com",
    "hotmail.com",
    "outlook.com",
    "yahoo.com",
    "icloud.com",
    "live.com",
    "msn.com",
    "contraloria.gob.pe",
]

MAPA_TYPOS_DOMINIO = {
    "gmai.com": "gmail.com",
    "gmal.com": "gmail.com",
    "gmial.com": "gmail.com",
    "gmail.con": "gmail.com",
    "gmail.co": "gmail.com",
    "gmail.cm": "gmail.com",
    "gmail.comm": "gmail.com",

    "hotmai.com": "hotmail.com",
    "hotmial.com": "hotmail.com",
    "homtail.com": "hotmail.com",
    "hotmail.con": "hotmail.com",
    "hotmail.co": "hotmail.com",
    "hotmail.cm": "hotmail.com",
    "hotmail.comm": "hotmail.com",

    "outlok.com": "outlook.com",
    "outloo.com": "outlook.com",
    "outlook.con": "outlook.com",

    "yaho.com": "yahoo.com",
    "yahoo.con": "yahoo.com",
}

In [15]:
def limpiar_email(valor):
    if pd.isna(valor):
        return pd.NA
    
    s = str(valor).strip().lower()
    
    # Quitar espacios internos
    s = re.sub(r"\s+", "", s)
    
    # Correcciones frecuentes
    s = s.replace(",", ".")
    s = s.replace("..", ".")
    
    # Si no tiene arroba, no se puede corregir con seguridad
    if "@" not in s:
        return s
    
    partes = s.split("@")
    
    # Si tiene más de un @, dejamos el primero como usuario y el último como dominio
    usuario = partes[0]
    dominio = partes[-1]
    
    # Quitar puntos raros al inicio/final
    usuario = usuario.strip(".")
    dominio = dominio.strip(".")
    
    # Corrección directa por mapa
    if dominio in MAPA_TYPOS_DOMINIO:
        dominio = MAPA_TYPOS_DOMINIO[dominio]
    else:
        # Corrección aproximada solo para dominios comunes
        match = get_close_matches(dominio, DOMINIOS_CORRECTOS, n=1, cutoff=0.88)
        if match:
            dominio = match[0]
    
    return f"{usuario}@{dominio}"

In [16]:
df_propietarios["email_clean"] = df_propietarios["email"].apply(limpiar_email)

In [17]:
patron_email = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

df_propietarios["email_ok"] = (
    df_propietarios["email_clean"]
    .astype("string")
    .str.match(patron_email, na=False)
)

In [18]:
df_propietarios.loc[
    df_propietarios["email"].astype("string").str.lower() != df_propietarios["email_clean"].astype("string"),
    ["email", "email_clean", "email_ok"]
].head(30)

,email,email_clean,email_ok
8050,melissagarciapl@gmai.com,melissagarciapl@gmail.com,True
40802,andresrivasplatacastro@gmal.com,andresrivasplatacastro@gmail.com,True
50126,arovi50@hotmil.com,arovi50@hotmail.com,True
57888,notiene@gmai1l.com,notiene@gmail.com,True
63651,christian.pablo24@gmail.con,christian.pablo24@gmail.com,True
65376,kiararengifo3@gmail.con,kiararengifo3@gmail.com,True
68877,jenny@gmaiil.com,jenny@gmail.com,True
88254,1226652@hotmail.colm,1226652@hotmail.com,True
93428,anamelbaquisperegular@gmail.om,anamelbaquisperegular@gmail.com,True
126588,emiliaicochea@gmauil.com,emiliaicochea@gmail.com,True


In [19]:
df_propietarios.loc[
    df_propietarios["email"].astype("string").str.lower() != df_propietarios["email_clean"].astype("string"),
    ["email", "email_clean", "email_ok"]
].head(30)

,email,email_clean,email_ok
8050,melissagarciapl@gmai.com,melissagarciapl@gmail.com,True
40802,andresrivasplatacastro@gmal.com,andresrivasplatacastro@gmail.com,True
50126,arovi50@hotmil.com,arovi50@hotmail.com,True
57888,notiene@gmai1l.com,notiene@gmail.com,True
63651,christian.pablo24@gmail.con,christian.pablo24@gmail.com,True
65376,kiararengifo3@gmail.con,kiararengifo3@gmail.com,True
68877,jenny@gmaiil.com,jenny@gmail.com,True
88254,1226652@hotmail.colm,1226652@hotmail.com,True
93428,anamelbaquisperegular@gmail.om,anamelbaquisperegular@gmail.com,True
126588,emiliaicochea@gmauil.com,emiliaicochea@gmail.com,True


## 3) Procesos de Separación

In [20]:
carpeta = "../data/raw/"
file = carpeta + "procesos.parquet"

In [21]:
df_raw = pd.read_parquet(file)
df_raw.head()

,codigo_proyecto,nombre_proyecto,tipo_unidad_principal,codigo_unidad,total_unidades,codigo_unidades_asignadas,nombres_cliente,apellidos_cliente,documento_cliente,origen_proforma,fecha_proforma,codigo_proforma,numero_contrato,fecha_contrato,modalidad_contrato,moneda,tipo_cambio,tipo_financiamiento,banco,situacion_legal,documento_representante,nombres_usuario,username,precio_base_proforma,descuento_venta,precio_venta,aprobador_descuento,nombre,premios,fecha_inicio,fecha_fin,fecha_expiracion,fecha_impresion_contrato,nombre_flujo,estado,completado,total_pagado,total_pendiente,fecha_analisis,fecha_nif,estado_nif,utm_source,utm_medium,utm_campaign,utm_term,utm_content,documento_conyuge,documento_copropietarios,flujo_anulacion,fecha_anulacion,codigo_externo_venta,id,tipo,fecha_actualizacion,codigo_externo_minuta,flujo_error,momento_caida,tipo_cronograma,estado_contrato,devolucion,fecha_devolucion,excedente,observacion_devolucion,motivo_caida,nombres_usuario_aprobador,username_aprobador,cliente_id,usuario_creador,username_creador,usuario_separacion,codigo_externo_entrega,fecha_minuta,proforma_id,penalidad,proceso_anulacion,codigo_externo_anulacion_venta,terminado,paso_actual,estado_personalizado
0,EEUU,Edificio Urbanzen,departamento flat,EEUU-302,2,EEUU-E19,Jan Henri,Larsen,48997555,manual,2025-09-16,2025-05785,NaN,None,Compromiso,PEN,1.000000,hipotecario,NaN,Conyugal,17896123,Yomira Pacheco,ypacheco,712460.00,77460.00,635000.00,ldelsolar,Entrega,None,2026-01-26,2026-01-30,None,None,Proceso de entrega,Activo,Completado,0.00,0.00,2026-01-30,None,NaN,None,None,None,None,None,17896123,NaN,NaN,None,None,21,ENTREGA,2026-01-30,None,False,NaN,NaN,NaN,0.00,None,0.0,NaN,None,NaN,NaN,121739,JOSUE QUIROGA NAVARRO,jquiroga,NaN,None,None,26094,0.00,al iniciar,None,None,Proceso Terminado,None
1,EEUU,Edificio Urbanzen,departamento flat,EEUU-1501,2,EEUU-E26,Katherine Jessy,Bendezú Delgadillo,45347923,manual,2024-03-19,2024-00497,NaN,None,Compromiso,PEN,1.000000,ahorro,Banco de Crédito del Perú (BCP),Propietario Solo,NaN,Nelly Román Villavicencio,nroman,659509.20,72744.51,586764.69,evillanueva,Entrega,None,2026-03-17,2026-03-17,None,None,Proceso de entrega,Activo,Completado,0.00,0.00,2026-03-17,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,68,ENTREGA,2026-03-17,None,False,NaN,NaN,NaN,0.00,None,0.0,NaN,None,NaN,NaN,43624,JOSUE QUIROGA NAVARRO,jquiroga,NaN,None,None,16905,0.00,al iniciar,None,None,Proceso Terminado,None
2,EEUU,Edificio Urbanzen,departamento flat,EEUU-1004,1,NaN,Daniel Ivan,Rivera Perez,07972081,manual,2025-06-08,2025-03322,NaN,None,Compromiso,PEN,1.000000,hipotecario,NaN,Propietario Solo,NaN,Yomira Pacheco,ypacheco,611003.69,66003.69,545000.00,evillanueva,Entrega,None,2026-02-02,2026-01-31,None,None,Proceso de entrega,Activo,Completado,0.00,0.00,2026-01-31,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,27,ENTREGA,2026-02-02,None,False,NaN,NaN,NaN,0.00,None,0.0,NaN,None,NaN,NaN,104639,JOSUE QUIROGA NAVARRO,jquiroga,NaN,None,None,23631,0.00,al iniciar,None,None,Proceso Terminado,None
3,TZ,Tizón y Bueno,departamento flat,TZ-207,2,TZ-10,LAURA CONSUELO,VALDIVIA PACHECO,07261199,manual,2025-10-25,2025-06988,NaN,None,Compromiso,PEN,1.000000,contado,Banco de Crédito del Perú (BCP),Propietario Solo,NaN,Percy Soto,psoto,609782.78,54096.78,555686.00,psoto,Entrega,None,2025-12-11,None,None,None,Proceso de entrega,Activo,No Completado,0.00,0.00,None,None,NaN,None,None,None,None,None,NaN,NaN,NaN,None,None,7,ENTREGA,2025-12-11,None,False,NaN,NaN,NaN,0.00,None,0.0,NaN,None,NaN,NaN,117871,JOSUE QUIROGA NAVARRO,jquiroga,NaN,None,None,27297,0.00,al iniciar,None,None,Encuestas de propietarios,None
4,EEUU,Edificio Urbanzen,departamento flat,EEUU-802,2,EEUU-E23,NADIA,GUERRA PONCE DE LEON,46230122,manual,2024-07-16,2024-01877,NaN,None,Compromiso,PEN,1.000000,hipotecario,Banco de Crédito del Perú (BCP),Propietario Solo,NaN,Nelly Román Villavicencio,nroman,611070.02,56237.00,554833.02,nroman,Entrega,None,2026-05-19,2026-05-18,None,None,Proces

### Separaciones

In [22]:
df_raw['nombre_proyecto'].value_counts()

nombre_proyecto
Edificio Cuba Connect           582
Tizón y Bueno                   532
Edificio Santa Cruz Infinite    400
Edificio Saenz                  376
Torre Nápoles                   358
Edificio Mariategui             336
Edificio Urbanzen               313
Modena                          313
Alicanto                        295
Sialia                          294
Fenix                           284
Matera                          224
Capadocia                       207
Los Jardines de Urteaga         185
Edificio Valdizan                91
Tradiciones Prime                12
Edificio Unique                   4
Name: count, dtype: int64

In [23]:
df_separaciones = df_raw[df_raw['nombre_proyecto']=="Fenix"]
df_separaciones = df_separaciones[df_separaciones['estado']=="Activo"]
df_separaciones = df_separaciones[df_separaciones['nombre']=="Separacion"]
print(f'Total Separaciones Activas: {df_separaciones.shape[0]} clientes')
col_contacto = ['nombre_proyecto','tipo_unidad_principal', 'codigo_unidad' , 'documento_cliente', 'nombres_usuario']
df_separaciones = df_separaciones[col_contacto]
#df_separaciones
df_separaciones.sample(80)


Total Separaciones Activas: 86 clientes


,nombre_proyecto,tipo_unidad_principal,codigo_unidad,documento_cliente,nombres_usuario
2639,Fenix,departamento flat,FX-503,06010468,Ana Lucia DelSolar
2652,Fenix,estacionamiento con depósito,FX-E14,06246788,Nelly Román Villavicencio
1569,Fenix,departamento flat,FX-103,07240945,Nelly Román Villavicencio
496,Fenix,estacionamiento con depósito,FX-E10,71300365,Yomira Pacheco
4712,Fenix,departamento flat,FX-1405,25705638,Luis Enrique Del Solar Cáceres
...,...,...,...,...,...
278,Fenix,departamento flat,FX-1205,70767841,Ivan La Riva
3853,Fenix,departamento flat,FX-1605,29352434,Edwing Soller
4394,Fenix,departamento flat,FX-904,07990374,Joceline Arizabal Corimanya
1843,Fenix,departamento flat,FX-604,10350646,Ivan La Riva


In [24]:
df_separaciones['tipo_unidad_principal'].value_counts()

tipo_unidad_principal
departamento flat               71
estacionamiento con depósito     8
estacionamiento simple           6
departamento duplex              1
Name: count, dtype: int64

In [25]:
df_separaciones.info()
#df_separaciones['nombre'].value_counts()

<class 'pandas.DataFrame'>
Index: 86 entries, 205 to 4735
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   nombre_proyecto        86 non-null     str  
 1   tipo_unidad_principal  86 non-null     str  
 2   codigo_unidad          86 non-null     str  
 3   documento_cliente      86 non-null     str  
 4   nombres_usuario        86 non-null     str  
dtypes: str(5)
memory usage: 8.8 KB


# Merge con Sperant

## 1) Main

In [26]:
print(f'Total Entregas Pestaña 1: {df_entregas.shape[0]} clientes')
df_entregas.sample(2)

Total Entregas Pestaña 1: 39 clientes


,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.
31,27.0,DEPARTAMENTO,1203,MELGAR SALINAS ANTONY JOSUHE,71251490,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31 Mar 2026,05 May 2026,07 Ago 2026,2:00 pm,NaN,NaN
18,NaN,ESTACIONAMIENTO,47,RODAS BARDALES VICTOR RICARDO,44507047,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24 Sep 2025,26 Mar 2026,31 Jul 2026,4:00 pm,NaN,NaN


In [27]:
print(f'Total Entregas Pestaña 2: {df_procesos.shape[0]} clientes')
df_procesos.sample(2)


Total Entregas Pestaña 2: 66 clientes


,#,Tipo,N° Unidad,Comprador,% Avance,F. Separación,unidad_key,codigo_unidad
4,5,ESTACIONAMIENTO,32,CAMAHUALI HUAMAN NANCY LIZ,0.000,NaN,32,FX-E32
21,22,ESTACIONAMIENTO,33,FERNANDEZ GOMEZ FERNANDO,0.217,23 Jun 2025,33,FX-E33


### 2) Right

In [28]:
print(f'Total Propietarios: {df_propietarios.shape[0]} clientes')
df_propietarios.sample(2)

Total Propietarios: 1301 clientes


,documento,celulares,email,telefono,nombres,apellidos,celular_clean,celular_len,celular_ok,email_clean,email_ok
72123,09315922,+51999888062,lmayvar@hotmail.com,,LUZ MARLENI,AYVAR MANCILLA,999888062,9,True,lmayvar@hotmail.com,True
128792,auto-135719,+51997715074,SILVI@GMAIL.COM,,SILVIA,ANGULO VERA,997715074,9,True,silvi@gmail.com,True


### Dni para 2 propietarios diferente

In [29]:
df_propietarios["documento_key"] = (
    df_propietarios["documento"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(8)
)

df_propietarios.loc[
    df_propietarios["documento_key"] == "25493893",
    ["documento", "documento_key"]
] = ["45063421", "45063421"]

## 3) Merge

### PESTAÑA 1

In [30]:
# df_propietarios = documento
# df_entregas_completo = df_entregas.merge(df_propietarios, how='left', on='DNI')

In [31]:
import pandas as pd

# Copias para no tocar los dataframes originales
entregas = df_entregas.copy()
propietarios = df_propietarios.copy()

# 1) Normalizar llave DNI / documento
def limpiar_documento(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)      # elimina .0 si vino de Excel
        .str.replace(r"\D", "", regex=True)        # deja solo números
        .replace("", pd.NA)
    )

entregas["dni_key"] = limpiar_documento(entregas["DNI"])
propietarios["documento_key"] = limpiar_documento(propietarios["documento"])

# 2) Opcional pero recomendado: quedarte solo con documentos válidos de 8 dígitos
entregas.loc[entregas["dni_key"].str.len() != 8, "dni_key"] = pd.NA
propietarios.loc[propietarios["documento_key"].str.len() != 8, "documento_key"] = pd.NA

# 3) Evitar duplicados en propietarios para no multiplicar filas
cols_propietarios = [
    "documento_key",
    "celulares",
    "celular_clean",
    "email",
    "email_clean",
    "telefono",
    "nombres"
]

propietarios_merge = (
    propietarios[cols_propietarios]
    .dropna(subset=["documento_key"])
    .drop_duplicates(subset=["documento_key"], keep="first")
)

# 4) Left merge: mantiene todas las filas de entregas
df_entregas_merge = entregas.merge(
    propietarios_merge,
    how="left",
    left_on="dni_key",
    right_on="documento_key",
    validate="m:1",
    indicator=True
)

df_entregas_merge.head()

,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.,dni_key,documento_key,celulares,celular_clean,email,email_clean,telefono,nombres,_merge
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN,45816592,45816592,+51999237144,999237144,peraltav.lisbeth@gmail.com,peraltav.lisbeth@gmail.com,,Lisbeth Sofia,both
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN,07990374,07990374,+51970465167,970465167,matoskina@gmail.com,matoskina@gmail.com,,monica,both
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN,71300365,71300365,+51962298910,962298910,josuedv13@hotmail.com,josuedv13@hotmail.com,-,josue HAMILTON,both
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both


### Validaciones

In [32]:
df_entregas_merge["_merge"].value_counts()

_merge
both          39
left_only      0
right_only     0
Name: count, dtype: int64

In [33]:
df_entregas_merge[df_entregas_merge["_merge"] == "left_only"][
    ["DNI", "dni_key", "Comprador", "Tipo Unidad", "N° Unidad", "celular_clean"]
].head(20)

,DNI,dni_key,Comprador,Tipo Unidad,N° Unidad,celular_clean


In [34]:
df_entregas_merge

,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.,dni_key,documento_key,celulares,celular_clean,email,email_clean,telefono,nombres,_merge
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN,45816592,45816592,+51999237144,999237144,peraltav.lisbeth@gmail.com,peraltav.lisbeth@gmail.com,,Lisbeth Sofia,both
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN,07990374,07990374,+51970465167,970465167,matoskina@gmail.com,matoskina@gmail.com,,monica,both
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN,71300365,71300365,+51962298910,962298910,josuedv13@hotmail.com,josuedv13@hotmail.com,-,josue HAMILTON,both
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both
5,5.0,DEPARTAMENTO,804,ROMO AGUIRRE LUIGUI ROLANDO,74827647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20 Dic 2024,02 Oct 2025,22 Jul 2026,9:00 am,NaN,NaN,74827647,74827647,+51932750569,932750569,romoluigui7@gmail.com,romoluigui7@gmail.com,-,Luigui ROLANDO,both
6,6.0,DEPARTAMENTO,704,LLANQUE FRAQUITA MANINO HELAMAN,40654929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Jul 2025,14 Oct 2025,22 Jul 2026,11:00 am,NaN,NaN,40654929,40654929,+51952852015,952852015,marinoii@hotmail.com,marinoii@hotmail.com,,Mariano Helaman,both
7,7.0,DEPARTAMENTO,1705,BULEJE CALDERON WILDER WILFREDO,71477258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,30 Oct 2025,22 Jul 2026,2:00 pm,NaN,NaN,71477258,71477258,+51986687399,986687399,wilderbuleje@hotmail.com,wilderbuleje@hotmail.com,,Wilder Wilfredo,both
8,8.0,DEPARTAMENTO,303,ALDERETE SOLIS SADITH PATRICIA,46137521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,31 Oct 2025,22 Jul 2026,4:00 pm,NaN,NaN,46137521,46137521,+51987469664,987469664,spatricia.2109@GMAIL.COM,spatricia.2109@gmail.com,987469664,SADITH PATRICIA,both
9,9.0,DEPARTAMENTO,802,BELLIDO PANTOJA ROSA MARIA,73097248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21 Ago 2025,24 Dic 2025,24 Jul 2026,9:00 am,NaN,NaN,73097248,73097248,+51965451936,965451936,rosabellidop23@gmail.com,rosabellidop23@gmail.com,,Rosa maria,both


In [35]:
df_separaciones

,nombre_proyecto,tipo_unidad_principal,codigo_unidad,documento_cliente,nombres_usuario
205,Fenix,estacionamiento simple,FX-E36,10560125,Nelly Román Villavicencio
206,Fenix,estacionamiento con depósito,FX-E29,41575485,Ana Lucia DelSolar
278,Fenix,departamento flat,FX-1205,70767841,Ivan La Riva
458,Fenix,departamento flat,FX-1404,10560125,Ana Lucia DelSolar
485,Fenix,departamento flat,FX-1504,49034354,Ivan La Riva
...,...,...,...,...,...
4536,Fenix,departamento flat,FX-402,25731745,Nelly Román Villavicencio
4684,Fenix,departamento flat,FX-905,20565889657,Ana Lucia DelSolar
4712,Fenix,departamento flat,FX-1405,25705638,Luis Enrique Del Solar Cáceres
4724,Fenix,departamento flat,FX-1705,71477258,Antonio Martinez


In [36]:
import pandas as pd

# Copias para no tocar los dataframes originales
entregas = df_entregas_merge.copy()
propietarios = df_separaciones.copy()

# 1) Normalizar llave DNI / documento
def limpiar_documento(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)      # elimina .0 si vino de Excel
        .str.replace(r"\D", "", regex=True)        # deja solo números
        .replace("", pd.NA)
    )

# Crear llaves limpias
entregas["dni_key"] = limpiar_documento(entregas["DNI"])
propietarios["documento_key"] = limpiar_documento(propietarios["documento_cliente"])

# 2) Quedarte solo con documentos válidos de 8 dígitos
entregas.loc[entregas["dni_key"].str.len() != 8, "dni_key"] = pd.NA
propietarios.loc[propietarios["documento_key"].str.len() != 8, "documento_key"] = pd.NA

# 3) Evitar duplicados en propietarios para no multiplicar filas
cols_propietarios = [
    "documento_key",
    "documento_cliente",
    "nombres_usuario", 
    "codigo_unidad"
]

propietarios_merge = (
    propietarios[cols_propietarios]
    .dropna(subset=["documento_key"])
    .drop_duplicates(subset=["documento_key"], keep="first")
)

# 4) Left merge: mantiene todas las filas de entregas
df_entregas_2_merge = entregas.merge(
    propietarios_merge,
    how="left",
    left_on="dni_key",
    right_on="documento_key",
    validate="m:1",
    indicator="merge_propietarios"
)

df_entregas_2_merge.head()

,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.,dni_key,documento_key_x,celulares,celular_clean,email,email_clean,telefono,nombres,_merge,documento_key_y,documento_cliente,nombres_usuario,codigo_unidad,merge_propietarios
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN,45816592,45816592,+51999237144,999237144,peraltav.lisbeth@gmail.com,peraltav.lisbeth@gmail.com,,Lisbeth Sofia,both,45816592,45816592,Edwing Soller,FX-1704,both
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN,07990374,07990374,+51970465167,970465167,matoskina@gmail.com,matoskina@gmail.com,,monica,both,07990374,07990374,Joceline Arizabal Corimanya,FX-904,both
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN,71300365,71300365,+51962298910,962298910,josuedv13@hotmail.com,josuedv13@hotmail.com,-,josue HAMILTON,both,71300365,71300365,Yomira Pacheco,FX-E10,both
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both,47160251,47160251,Ana Lucia DelSolar,FX-1702,both
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both,47160251,47160251,Ana Lucia DelSolar,FX-1702,both


## PESTAÑA 2

In [37]:
df_procesos.sample(6)

,#,Tipo,N° Unidad,Comprador,% Avance,F. Separación,unidad_key,codigo_unidad
36,37,ESTACIONAMIENTO,43,MELGAR SALINAS ANTONY JOSUHE,0.452,22 Abr 2026,43,FX-E43
15,16,ESTACIONAMIENTO,27,CUNYAS SANCHEZ KEVIN MAYCOL,0.000,NaN,27,FX-E27
22,23,ESTACIONAMIENTO,14,FIGUEROA SILVA DE CORILLOCLLA FANI MARMEL,0.163,27 Jul 2025,14,FX-E14
61,62,DEPARTAMENTO,901,VILA HUANCA GLADYS CLAUDIA,0.097,20 May 2026,901,FX-901
52,53,DEPARTAMENTO,1202,SANCHEZ CACERES ROSAURA,0.500,23 Jul 2025,1202,FX-1202
2,3,DEPARTAMENTO,1201,CACERES VALVERDE HENRY ENRIQUE,0.003,16 Jun 2026,1201,FX-1201


In [38]:
df_separaciones

,nombre_proyecto,tipo_unidad_principal,codigo_unidad,documento_cliente,nombres_usuario
205,Fenix,estacionamiento simple,FX-E36,10560125,Nelly Román Villavicencio
206,Fenix,estacionamiento con depósito,FX-E29,41575485,Ana Lucia DelSolar
278,Fenix,departamento flat,FX-1205,70767841,Ivan La Riva
458,Fenix,departamento flat,FX-1404,10560125,Ana Lucia DelSolar
485,Fenix,departamento flat,FX-1504,49034354,Ivan La Riva
...,...,...,...,...,...
4536,Fenix,departamento flat,FX-402,25731745,Nelly Román Villavicencio
4684,Fenix,departamento flat,FX-905,20565889657,Ana Lucia DelSolar
4712,Fenix,departamento flat,FX-1405,25705638,Luis Enrique Del Solar Cáceres
4724,Fenix,departamento flat,FX-1705,71477258,Antonio Martinez


In [39]:
df_propietarios

,documento,celulares,email,telefono,nombres,apellidos,celular_clean,celular_len,celular_ok,email_clean,email_ok,documento_key
48,25617270,+51996715677,ncolichon@yahoo.com,,Nicolas Armando,Colichon Chirinos,996715677,9,True,ncolichon@yahoo.com,True,25617270
99,auto-292836,+51996050820,lilopezc@hotmail.com,+51996050820,Li Giovanna,Lopez,996050820,9,True,lilopezc@hotmail.com,True,00292836
130,20121877,+51937818236,yesys123443@gmail.com,,YESENIA MARLENY,PALOMINO VICTORIO,937818236,9,True,yesys123443@gmail.com,True,20121877
131,09614353,+51924719407,schavezs_02@hotmail.com,NaN,Sonia Silvia,Chavez Sipiran,924719407,9,True,schavezs_02@hotmail.com,True,09614353
142,07766389,+51959762690,monicapatricia2708@hotmail.com,,Monica Patricia,Chuco Linares,959762690,9,True,monicapatricia2708@hotmail.com,True,07766389
...,...,...,...,...,...,...,...,...,...,...,...,...
137573,40078833,+51985392292,ESTHER_ALVITES@HOTMAIL.COM,985392292,ESTHER,ALVITES COTRINA VIUDA DE MARIN,985392292,9,True,esther_alvites@hotmail.com,True,40078833
137575,18198498,+51959662977,jroxci2015@gmail.com,,JANETH ROXANA,CHUAN IBAÑEZ,959662977,9,True,jroxci2015@gmail.com,True,18198498
137576,70825918,+51949456650,victorzr8@gmail.com,,VICTOR ANDRES,ZAVALETA DE LOS RIOS,949456650,9,True,victorzr8@gmail.com,True,70825918
137599,16776183,+51923440342,carmen7_@outlook.com.pe,,César Alberto,García Espinoza,923440342,9,True,carmen7_@outlook.com,True,16776183


In [40]:
import pandas as pd

# Copias para no tocar los dataframes originales
procesos     = df_procesos.copy()
procesos_separaciones = df_separaciones.copy()

# 1) Normalizar llave DNI / documento
def limpiar_documento(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)      # elimina .0 si vino de Excel
        .str.replace(r"\D", "", regex=True)        # deja solo números
        .replace("", pd.NA)
    )

procesos["unidad_key"] = limpiar_documento(procesos["codigo_unidad"])
procesos_separaciones["unidad_key"] = limpiar_documento(procesos_separaciones["codigo_unidad"])

# 2) Opcional pero recomendado: quedarte solo con documentos válidos de 8 dígitos
#procesos.loc[procesos["unidad_key"].str.len() != 8, "unidad_key"] = pd.NA
#propietarios.loc[propietarios["unidad_key"].str.len() != 8, "unidad_key"] = pd.NA

# 3) Evitar duplicados en propietarios para no multiplicar filas
cols_propietarios = [
    "unidad_key",
    "nombres_usuario",
    "documento_cliente"#,
    # "email",
    # "email_clean",
    # "telefono",
    # "nombres"
]

propietarios_merge = (
    procesos_separaciones[cols_propietarios]
    .dropna(subset=["unidad_key"])
    .drop_duplicates(subset=["unidad_key"], keep="first")
)

# 4) Left merge: mantiene todas las filas de entregas
df_procesos_merge = procesos.merge(
    propietarios_merge,
    how="left",
    left_on="unidad_key",
    right_on="unidad_key",
    validate="m:1",
    indicator=True
)

df_procesos_merge.head()

,#,Tipo,N° Unidad,Comprador,% Avance,F. Separación,unidad_key,codigo_unidad,nombres_usuario,documento_cliente,_merge
0,1,ESTACIONAMIENTO,7,BELLIDO PANTOJA ROSA MARIA,0.770,24 Jun 2025,7,FX-E7,Joceline Arizabal Corimanya,73097248,both
1,2,DEPARTAMENTO,402,BLAZ MARIÑO FRANK,0.100,06 Jun 2026,402,FX-402,Nelly Román Villavicencio,25731745,both
2,3,DEPARTAMENTO,1201,CACERES VALVERDE HENRY ENRIQUE,0.003,16 Jun 2026,1201,FX-1201,Percy Soto,70459143,both
3,4,DEPARTAMENTO,903,CAMAHUALI HUAMAN NANCY LIZ,0.249,06 Abr 2025,903,FX-903,Sonia Paredes,41297302,both
4,5,ESTACIONAMIENTO,32,CAMAHUALI HUAMAN NANCY LIZ,0.000,NaN,32,FX-E32,NaN,NaN,left_only


In [41]:
import pandas as pd

# Copias para no tocar los dataframes originales
entregas = df_procesos_merge.copy()
propietarios = df_propietarios.copy()

# 1) Normalizar llave DNI / documento
def limpiar_documento(serie):
    return (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)      # elimina .0 si vino de Excel
        .str.replace(r"\D", "", regex=True)        # deja solo números
        .replace("", pd.NA)
    )

entregas["dni_key"] = limpiar_documento(entregas["documento_cliente"])
propietarios["documento_key"] = limpiar_documento(propietarios["documento"])

# 2) Opcional pero recomendado: quedarte solo con documentos válidos de 8 dígitos
entregas.loc[entregas["dni_key"].str.len() != 8, "dni_key"] = pd.NA
propietarios.loc[propietarios["documento_key"].str.len() != 8, "documento_key"] = pd.NA

# 3) Evitar duplicados en propietarios para no multiplicar filas
cols_propietarios = [
    "documento_key",
    "celulares",
    "celular_clean",
    "email",
    "email_clean",
    "telefono",
    "nombres"
]

propietarios_merge = (
    propietarios[cols_propietarios]
    .dropna(subset=["documento_key"])
    .drop_duplicates(subset=["documento_key"], keep="first")
)

# 4) Left merge: mantiene todas las filas de entregas
df_entregas_merge = entregas.merge(
    propietarios_merge,
    how="left",
    left_on="dni_key",
    right_on="documento_key",
    validate="m:1",
    #indicator=True
)

df_entregas_merge.to_excel('En Proceso.xlsx', index=False)
df_entregas_merge.head()

,#,Tipo,N° Unidad,Comprador,% Avance,F. Separación,unidad_key,codigo_unidad,nombres_usuario,documento_cliente,_merge,dni_key,documento_key,celulares,celular_clean,email,email_clean,telefono,nombres
0,1,ESTACIONAMIENTO,7,BELLIDO PANTOJA ROSA MARIA,0.770,24 Jun 2025,7,FX-E7,Joceline Arizabal Corimanya,73097248,both,73097248,73097248,+51965451936,965451936,rosabellidop23@gmail.com,rosabellidop23@gmail.com,,Rosa maria
1,2,DEPARTAMENTO,402,BLAZ MARIÑO FRANK,0.100,06 Jun 2026,402,FX-402,Nelly Román Villavicencio,25731745,both,25731745,25731745,+51976362387,976362387,FRANKBLAZ01@GMAIL.COM,frankblaz01@gmail.com,,FRANK
2,3,DEPARTAMENTO,1201,CACERES VALVERDE HENRY ENRIQUE,0.003,16 Jun 2026,1201,FX-1201,Percy Soto,70459143,both,70459143,70459143,+51995966779,995966779,henrycaceresv@gmail.com,henrycaceresv@gmail.com,,HENRY ENRIQUE
3,4,DEPARTAMENTO,903,CAMAHUALI HUAMAN NANCY LIZ,0.249,06 Abr 2025,903,FX-903,Sonia Paredes,41297302,both,41297302,41297302,+51972306528,972306528,lcamahualih@gmail.com,lcamahualih@gmail.com,,Nancy Liz
4,5,ESTACIONAMIENTO,32,CAMAHUALI HUAMAN NANCY LIZ,0.000,NaN,32,FX-E32,NaN,NaN,left_only,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN


## Entrega Listo

In [42]:
print(f'Total Entregas Pestaña 1: {df_entregas_2_merge.shape[0]} clientes')
col_contacto = ['nombre_proyecto','tipo_unidad_principal', 'codigo_unidad' , 'documento_cliente', 'nombres_usuario']
df_separaciones = df_separaciones[col_contacto]
df_entregas_2_merge.to_excel('Entregas.xlsx', index=False)
df_entregas_2_merge.sample(15)

Total Entregas Pestaña 1: 39 clientes


,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.,dni_key,documento_key_x,celulares,celular_clean,email,email_clean,telefono,nombres,_merge,documento_key_y,documento_cliente,nombres_usuario,codigo_unidad,merge_propietarios
5,5.0,DEPARTAMENTO,804,ROMO AGUIRRE LUIGUI ROLANDO,74827647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20 Dic 2024,02 Oct 2025,22 Jul 2026,9:00 am,NaN,NaN,74827647,74827647,+51932750569,932750569,romoluigui7@gmail.com,romoluigui7@gmail.com,-,Luigui ROLANDO,both,74827647,74827647,Yomira Pacheco,FX-E9,both
29,25.0,DEPARTAMENTO,1405,ALEJOS SILVA KELVIN ARTURO,25705638,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Ago 2025,29 Abr 2026,07 Ago 2026,9:00 am,NaN,NaN,25705638,25705638,+51953661622,953661622,arturo.alejos21@gmail.com,arturo.alejos21@gmail.com,,Kelvin Arturo,both,25705638,25705638,Luis Enrique Del Solar Cáceres,FX-1405,both
15,14.0,DEPARTAMENTO,1305,GUZMAN DAVILA ANDREA NICOL,72757053,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Jun 2025,05 Mar 2026,31 Jul 2026,11:00 am,NaN,NaN,72757053,72757053,+51973723315,973723315,aguzmandavila.agd@gmail.com,aguzmandavila.agd@gmail.com,,Andrea Nicol,both,72757053,72757053,Joceline Arizabal Corimanya,FX-1305,both
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN,45816592,45816592,+51999237144,999237144,peraltav.lisbeth@gmail.com,peraltav.lisbeth@gmail.com,,Lisbeth Sofia,both,45816592,45816592,Edwing Soller,FX-1704,both
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both,47160251,47160251,Ana Lucia DelSolar,FX-1702,both
35,30.0,DEPARTAMENTO,305,DE LA CRUZ MERCADO DIANA VANESSA,71221947,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17 Nov 2025,28 May 2026,10 Ago 2026,11:00 am,NaN,NaN,71221947,71221947,+51941130864,941130864,dianavanessa7696@gmail.com,dianavanessa7696@gmail.com,,Diana Vanessa,both,71221947,71221947,Antonio Martinez,FX-305,both
28,NaN,ESTACIONAMIENTO,29,RUIZ MARINA RICARDO,41575485,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25 Oct 2025,27 Abr 2026,05 Ago 2026,4:00 pm,NaN,NaN,41575485,41575485,+51965077971,965077971,ricardomarima1982@gmail.com,ricardomarima1982@gmail.com,,Ricardo,both,41575485,41575485,Ana Lucia DelSolar,FX-E29,both
20,18.0,DEPARTAMENTO,601,JULIAN GOMEZ ROSA YASIDH,70361676,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,09 Jul 2025,31 Mar 2026,03 Ago 2026,11:00 am,NaN,NaN,70361676,70361676,+51963893559,963893559,rosidh02@gmail.com,rosidh02@gmail.com,,Rosa,both,70361676,70361676,Luis Peñaloza,FX-E16,both
23,21.0,DEPARTAMENTO,1605,LLERENA ALVAREZ VERONICA LUZ,29352434,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,06 Jul 2025,04 Mar 2026,05 Ago 2026,9:00 am,NaN,NaN,29352434,29352434,+51955286487,955286487,llerenaverito@hotmail.com,llerenaverito@hotmail.com,,Veronica Luz,both,29352434,29352434,Edwing Soller,FX-E35,both
24,NaN,ESTACIONAMIENTO,35,LLERENA ALVAREZ VERONICA LUZ,29352434,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,06 Jul 2025,15 Abr 2026,05 Ago 2026,9:00 am,NaN,NaN,29352434,29352434,+51955286487,955286487,llerenaverito@hotmail.com,llerenaverito@hotmail.com,,Veronica Luz,both,29352434,29352434,Edwing Soller,FX-E35,both


In [43]:
print(f'Total Entregas Pestaña 1: {df_entregas_merge.shape[0]} clientes')

df_entregas_merge.sample(15)
df_entregas_2_merge

Total Entregas Pestaña 1: 66 clientes


,#,Tipo Unidad,N° Unidad,Comprador,DNI,Coprop. 1,DNI Cop. 1,Coprop. 2,DNI Cop. 2,Coprop. 3,DNI Cop. 3,Coprop. 4,DNI Cop. 4,F. Separación,F. Cancelación,Fecha Entrega\nPropuesta,Horario,Fecha Entrega\nEfectiva,Obs.,dni_key,documento_key_x,celulares,celular_clean,email,email_clean,telefono,nombres,_merge,documento_key_y,documento_cliente,nombres_usuario,codigo_unidad,merge_propietarios
0,1.0,DEPARTAMENTO,1704,PERALTA VASQUEZ LISBETH SOFIA,45816592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16 Mar 2025,04 Sep 2025,20 Jul 2026,9:00 am,NaN,NaN,45816592,45816592,+51999237144,999237144,peraltav.lisbeth@gmail.com,peraltav.lisbeth@gmail.com,,Lisbeth Sofia,both,45816592,45816592,Edwing Soller,FX-1704,both
1,2.0,DEPARTAMENTO,904,MATTOS KINA MONICA PATRICIA,07990374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 Ago 2025,04 Sep 2025,20 Jul 2026,11:00 am,NaN,NaN,07990374,07990374,+51970465167,970465167,matoskina@gmail.com,matoskina@gmail.com,,monica,both,07990374,07990374,Joceline Arizabal Corimanya,FX-904,both
2,3.0,DEPARTAMENTO,1503,DEL VALLE VARILLAS JOSUE HAMILTON,71300365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Ago 2025,12 Sep 2025,20 Jul 2026,2:00 pm,NaN,NaN,71300365,71300365,+51962298910,962298910,josuedv13@hotmail.com,josuedv13@hotmail.com,-,josue HAMILTON,both,71300365,71300365,Yomira Pacheco,FX-E10,both
3,4.0,DEPARTAMENTO,1702,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04 Jun 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both,47160251,47160251,Ana Lucia DelSolar,FX-1702,both
4,NaN,ESTACIONAMIENTO,37,GARAY FLORES ROSMERY KAROL,47160251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12 Sep 2025,25 Sep 2025,20 Jul 2026,4:00 pm,NaN,NaN,47160251,47160251,+51955999063,955999063,rosmerygarayf@gmail.com,rosmerygarayf@gmail.com,,Rosmery,both,47160251,47160251,Ana Lucia DelSolar,FX-1702,both
5,5.0,DEPARTAMENTO,804,ROMO AGUIRRE LUIGUI ROLANDO,74827647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20 Dic 2024,02 Oct 2025,22 Jul 2026,9:00 am,NaN,NaN,74827647,74827647,+51932750569,932750569,romoluigui7@gmail.com,romoluigui7@gmail.com,-,Luigui ROLANDO,both,74827647,74827647,Yomira Pacheco,FX-E9,both
6,6.0,DEPARTAMENTO,704,LLANQUE FRAQUITA MANINO HELAMAN,40654929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18 Jul 2025,14 Oct 2025,22 Jul 2026,11:00 am,NaN,NaN,40654929,40654929,+51952852015,952852015,marinoii@hotmail.com,marinoii@hotmail.com,,Mariano Helaman,both,40654929,40654929,Antonio Martinez,FX-704,both
7,7.0,DEPARTAMENTO,1705,BULEJE CALDERON WILDER WILFREDO,71477258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,30 Oct 2025,22 Jul 2026,2:00 pm,NaN,NaN,71477258,71477258,+51986687399,986687399,wilderbuleje@hotmail.com,wilderbuleje@hotmail.com,,Wilder Wilfredo,both,71477258,71477258,Antonio Martinez,FX-1705,both
8,8.0,DEPARTAMENTO,303,ALDERETE SOLIS SADITH PATRICIA,46137521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30 Sep 2025,31 Oct 2025,22 Jul 2026,4:00 pm,NaN,NaN,46137521,46137521,+51987469664,987469664,spatricia.2109@GMAIL.COM,spatricia.2109@gmail.com,987469664,SADITH PATRICIA,both,46137521,46137521,Ivan La Riva,FX-303,both
9,9.0,DEPARTAMENTO,802,BELLIDO PANTOJA ROSA MARIA,73097248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21 Ago 2025,24 Dic 2025,24 Jul 2026,9:00 am,NaN,NaN,73097248,73097248,+51965451936,965451936,rosabellidop23@gmail.com,rosabellidop23@gmail.com,,Rosa maria,both,73097248,73097248,Joceline Arizabal Corimanya,FX-E7,both
